# Predicting Quarterly GDP Growth — Extended Project
## Building an Independent, Policy-Responsive Growth Model for Small Open Economies

**Skeleton Notebook** — instructions and structure only. Fill in every `# TODO` cell yourself.

### Project brief
You're an economist at a regional development bank covering small, open, emerging economies like Costa Rica — the kind of country whose growth is heavily shaped by tourism, commodity exports, external debt, and vulnerability to natural disasters. Your job is to build a **growth-driver model** that estimates quarterly real GDP growth from observable macroeconomic and structural indicators — independent of the IMF's own published forecast for that same period. Unlike a forecast (a single fixed number), your model can be **re-run under alternative scenarios** ("what if commodity prices fall 15%?", "what if a hurricane causes 5% of GDP in damage?"), which makes it useful for policy stress-testing even if it's not quite as accurate as the professionals' forecast on average. You're given `economy_indicators.csv` (1,278 country-quarter observations across 15 small Latin American/Caribbean economies, 29 raw columns).

This extended project introduces techniques the previous projects in this series didn't need: **accounting-style negative-number formatting** (parentheses instead of a minus sign — extremely common in real financial/economic data exports), a **structural-missingness pattern** (a column that's only meaningful, and only ever recorded, when a *different* column takes a specific value), and a hand-engineered **interaction feature** that lets a simple, interpretable linear model outperform default tree ensembles on a sparse-but-important effect.

> **This is a synthetic dataset built for a machine-learning exercise.** The country records, indicators, and forecasts are simulated, not real economic data. Nothing in this notebook is economic forecasting or policy advice — the scenario-simulation section demonstrates a methodology, not validated guidance for real policymaking.

### How to use this notebook
Each section has a short **Context** explaining *why* the step matters, then a **Task** list of exactly what to build. Use the **Cheat Sheet** notebook for syntax help and the **Background Theory** notebook for conceptual grounding. Don't peek at the Solutions notebook until you've attempted each section yourself.

### Success criteria for the whole project
- A cleaned, fully numeric feature matrix — including correctly parsing the **target itself**, which is stored in the same messy accounting format as the features
- A clear, written distinction between genuinely-missing data and structurally-absent data, each imputed appropriately
- At least 4 trained regression models compared fairly, including one with a hand-engineered interaction term
- A tuned final model with cross-validated performance estimates
- Two independent feature-importance views that agree on the top growth drivers
- A policy-scenario simulation showing how the model's growth estimate responds to hypothetical shocks


## Module 0 — Environment Setup

**Context.** Fixed seeds and consistent imports make your results comparable to the Solutions notebook.

**Task**
- Import `pandas`, `numpy`, `matplotlib.pyplot`, `seaborn`.
- Import `train_test_split`, `cross_val_score`, `RandomizedSearchCV` from `sklearn.model_selection`.
- Import `LinearRegression`, `Ridge` from `sklearn.linear_model`.
- Import `RandomForestRegressor`, `GradientBoostingRegressor` from `sklearn.ensemble`.
- Import `mean_absolute_error`, `mean_squared_error`, `r2_score` from `sklearn.metrics`.
- Set `RANDOM_STATE = 42` and use it everywhere a `random_state` argument exists.


In [ ]:
# TODO: imports and global constants


## Module 1 — Data Loading & Initial Inspection

**Context.** This dataset is styled after real macroeconomic data exports (World Bank/IMF-style), which have their own particular conventions.

**Task**
- Load `economy_indicators.csv` into `df`.
- Print `df.shape`, `df.head()`, `df.info()`.
- Print `df.describe(include='all').T`.
- Look closely at a few values in `gdp_growth_rate_pct` (your target!) and `inflation_rate_pct`. Do you notice a formatting convention for negative numbers that's different from a plain minus sign? (Hint: think about how accounting software often displays negative numbers.)


In [ ]:
# TODO: load data and inspect


## Module 2 — Data Quality Audit

**Context.** Two distinct kinds of "missingness" show up in this dataset, and treating them the same way would be a mistake:
1. **Random reporting gaps** — a smaller economy simply didn't report tourism arrivals, FDI, or education spending for a given quarter. These are genuinely unknown values.
2. **Structural absence** — `disaster_damage_pct_gdp` is only ever recorded when `natural_disaster_event` is `"Yes"`. When there's no disaster, there's no "unknown damage figure" to report — the true value is unambiguously zero, not missing.

**Task**
- Build a dtype/nunique/missing audit table.
- Check `df.duplicated().sum()`; drop confirmed duplicates.
- Look at `df['country'].unique()`. How many distinct-looking values are there, and how many *real* countries do you think that represents? (Hint: check for casing differences, leading/trailing spaces, and doubled internal spaces.)
- Write a cleaning step that strips whitespace, collapses repeated internal spaces, and normalizes casing for `country`.
- Investigate the `".."` values across the dataset (a real World Bank/IMF convention for missing data in exported indicator tables). Which columns use it?
- For `disaster_damage_pct_gdp` specifically: confirm that every `".."` corresponds to `natural_disaster_event == "No"`. Given that, decide how this column's missing values should be imputed, and justify why that's different from how you'd handle the *other* `".."`-bearing columns.


In [ ]:
# TODO: dtype audit table


In [ ]:
# TODO: duplicate check


In [ ]:
# TODO: investigate and clean the 'country' column's casing/whitespace inconsistencies


In [ ]:
# TODO: investigate '..' sentinel usage across columns; confirm the structural-missingness pattern for disaster_damage_pct_gdp


## Module 3 — Feature Engineering I: Accounting-Style Percentages (Including the Target)

**Context.** Most of this dataset's numeric indicators are stored as percentage strings using **accounting notation for negative values**: `"4.2%"` for a positive figure, but `"(1.7%)"` — parentheses, no minus sign — for a negative one. This convention needs to be parsed correctly before you can do anything numeric, and it applies to **your target column too** (`gdp_growth_rate_pct`), not just the features.

**Task**
- Write a helper function that takes a percentage string in this format and returns the correct signed float (handle both the `"X.X%"` and `"(X.X%)"` cases).
- Apply it to every percentage-formatted column, **including `gdp_growth_rate_pct` and `imf_next_year_growth_forecast_pct`**.
- Sanity-check a few known-negative rows (e.g. a row where `natural_disaster_event == "Yes"` and a large `disaster_damage_pct_gdp`) to confirm your parsing produced a sensible negative growth figure where you'd expect one.


In [ ]:
# TODO: write a parentheses-aware percentage parser


In [ ]:
# TODO: apply it to every percentage column, including the target and the IMF forecast column


## Module 4 — Feature Engineering II: Two Kinds of Missing Values

**Context.** Now that you've identified which columns have `".."` values and why, clean each type appropriately.

**Task**
- For `fdi_inflow_pct_gdp`, `education_spending_pct_gdp`, and `tourism_arrivals_millions` (random reporting gaps): create a `*_missing` flag column, then impute the sentinel rows with the column's median.
- For `disaster_damage_pct_gdp` (structural absence): impute missing values with **0**, not the median — and explain in a markdown cell why a flag column is *unnecessary* here (hint: you already have `natural_disaster_event` telling you the same thing).
- Clean `population` (strip commas and `" people"`) and `gdp_per_capita_usd` (strip `"$"` and commas); cast both to numeric.
- Convert `imf_program_active` and `natural_disaster_event` from `"Yes"/"No"` to `0/1`.


In [ ]:
# TODO: random-reporting-gap columns: flag + median impute


In [ ]:
# TODO: structural-absence column: impute with 0, explain why no flag is needed


In [ ]:
# TODO: clean population, gdp_per_capita_usd; convert Yes/No flags


## Module 5 — Feature Engineering III: Ordinal Encoding & a Hand-Crafted Interaction

**Context.** `income_group` has a natural order (Low → Lower middle → Upper middle → High income). Separately, domain knowledge suggests that a natural disaster hits a **high-debt** country's growth much harder than a low-debt country (less fiscal space to respond and rebuild) — an *interaction* effect that a plain linear model cannot represent unless you hand it the interaction explicitly.

**Task**
- Map `income_group` to an ordinal score.
- Create `disaster_debt_interaction = natural_disaster_event × government_debt_pct_gdp` (both should already be numeric from Modules 3–4).
- In one sentence, explain why a tree-based model doesn't strictly *need* this hand-crafted feature to represent the interaction, while a linear model does.


In [ ]:
# TODO: ordinal-encode income_group


In [ ]:
# TODO: create disaster_debt_interaction


## Module 6 — Advanced EDA & the Forecast-Leakage Audit

**Context.** `imf_next_year_growth_forecast_pct` is an expert forecast for the *same* period as your target — correlating strongly with it, but not as extremely as the market-quote traps in earlier projects in this series (professional GDP forecasting is genuinely hard, and forecasters are informed but imperfect).

**Task**
- Plot the distribution of your cleaned `gdp_growth_rate_pct`; compute its skewness.
- Build a correlation heatmap **including** `imf_next_year_growth_forecast_pct`. Compute the exact correlation with the target.
- Write one paragraph: even though this correlation is noticeably lower than the leakage traps in the previous two projects (sports betting, aircraft valuation), why does it still make sense to exclude this column from your feature set, given this project's specific goal?
- Compute VIF on your remaining numeric features.
- Boxplot the target by `natural_disaster_event` and by `currency_regime`.
- Check whether `disaster_debt_interaction` correlates with the target more strongly than `government_debt_pct_gdp` or `natural_disaster_event` do individually.


In [ ]:
# TODO: target distribution + skew


In [ ]:
# TODO: correlation heatmap INCLUDING the IMF forecast column


In [ ]:
# TODO: explicit correlation check + written leakage decision


In [ ]:
# TODO: VIF check


In [ ]:
# TODO: boxplots + interaction-feature correlation check


## Module 7 — Encoding Remaining Categorical Variables (Leakage-Safe)

**Context.** `region`, `quarter`, and `currency_regime` are low-cardinality nominal categoricals; `country` and `primary_export_sector` are higher-cardinality.

**Task**
- Using the `< 5 unique values` threshold, split the remaining (non-ordinal) categorical columns into one-hot vs. target-encode groups.
- Split into train/test **before** computing any target-encoding statistic.
- Fit target-encoding means on `y_train`/`X_train` only; apply to `X_test` with an unseen-category fallback.
- Confirm zero `NaN`s remain in `X_train`/`X_test`.


In [ ]:
# TODO: bucket remaining categoricals by cardinality


In [ ]:
# TODO: train/test split, then fit + apply leakage-safe target encoding


## Module 8 — Baseline & Linear Models (With and Without the Interaction Term)

**Task**
- Build a mean-predictor baseline; report MAE/RMSE/R² on the test set.
- Fit `LinearRegression` **without** `disaster_debt_interaction`; report the same three metrics.
- Fit a second `LinearRegression` **with** `disaster_debt_interaction` included; compare.
- Fit `Ridge` on the full feature set (with the interaction term).


In [ ]:
# TODO: mean-predictor baseline


In [ ]:
# TODO: LinearRegression without vs. with the interaction term


In [ ]:
# TODO: Ridge


## Module 9 — Tree-Ensemble Models

**Task**
- Train `RandomForestRegressor(random_state=RANDOM_STATE)` and `GradientBoostingRegressor(random_state=RANDOM_STATE)` with default hyperparameters (on the feature set that includes the interaction term, for a fair comparison).
- Collect every model into one comparison table sorted by MAE.
- Given that natural disasters occur in only a small fraction of rows, discuss why a tree ensemble might *not* automatically discover the disaster×debt interaction as reliably as you'd expect, even though trees can represent interactions in principle.


In [ ]:
# TODO: RandomForestRegressor, GradientBoostingRegressor


In [ ]:
# TODO: model comparison table + written discussion of the sparse-interaction problem


## Module 10 — Hyperparameter Tuning & Cross-Validation

**Task**
- Run 5-fold `cross_val_score` on your best model from Modules 8–9.
- Define a hyperparameter search space and run `RandomizedSearchCV` (`cv=5`).
- Report best params, best CV score, and the refit model's test-set performance.


In [ ]:
# TODO: cross_val_score on your leading candidate


In [ ]:
# TODO: RandomizedSearchCV, refit, evaluate on test set


## Module 11 — Evaluation & Diagnostics

**Task**
- Report MAE, RMSE, R², and MAPE for your final model (note: MAPE can behave oddly here too, since quarterly GDP growth can be very close to zero for some observations — flag this the same way the sports-betting project did).
- Plot predicted vs. actual growth with a `y=x` reference line.
- Plot residuals vs. predicted values, and separately, residuals for disaster-quarters vs. non-disaster-quarters — does your final model still systematically miss high-debt disaster cases?


In [ ]:
# TODO: MAE, RMSE, R2, MAPE (with a caveat on near-zero growth quarters)


In [ ]:
# TODO: predicted vs actual scatterplot + residual plots, split by disaster status


## Module 12 — Feature Importance

**Task**
- Plot the top 10 features by coefficient magnitude (linear) or impurity-based importance (tree model).
- Compute and plot permutation importance on the test set.
- Where does `disaster_debt_interaction` rank? Does it show up more clearly in one importance method than the other?


In [ ]:
# TODO: importance plot #1


In [ ]:
# TODO: permutation importance plot


## Module 13 — Policy Scenario Simulation

**Context.** This is the payoff for building an independent, input-driven model instead of just using the IMF's fixed forecast number: you can ask **"what if?"** questions a single forecast can't answer.

> Remember: this is a synthetic dataset and a simplified model. This section demonstrates a scenario-analysis methodology, not real economic forecasting or policy guidance.

**Task**
- Pick a specific country-quarter row from your test set to use as a baseline scenario (Costa Rica is a natural choice, given this project's framing).
- Simulate a **commodity price shock**: reduce `commodity_export_price_index` by 15% and re-predict growth. Report the change vs. the baseline prediction.
- Simulate a **tourism shock**: reduce `tourism_revenue_pct_gdp` by 30% (e.g. a pandemic-style disruption) and re-predict growth.
- Simulate a **disaster scenario**: set `natural_disaster_event` to 1, `disaster_damage_pct_gdp` to a plausible value (e.g. 5%), and recompute `disaster_debt_interaction` accordingly — compare the impact for the actual country's debt level vs. a hypothetical much-higher-debt version of the same country.
- Write two sentences on what this scenario capability offers that the IMF's single-point forecast doesn't.


In [ ]:
# TODO: pick a baseline row; define a re-predict helper that takes a modified feature row


In [ ]:
# TODO: commodity price shock scenario


In [ ]:
# TODO: tourism shock scenario


In [ ]:
# TODO: disaster scenario at two different debt levels


## Module 14 — Persistence & Inference

**Task**
- Save your final model with `joblib.dump`, along with encoding maps, ordinal maps, and the training column order.
- Write a function `predict_growth(raw_country_quarter_dict)` that takes a dictionary shaped like one row of the raw CSV (minus the IMF forecast column) and returns your model's growth estimate.
- Test it on 2–3 made-up country-quarters and sanity-check the outputs are plausible.


In [ ]:
# TODO: persist model + encoders


In [ ]:
# TODO: predict_growth(raw_country_quarter_dict) function + sanity checks


## Module 15 — Conclusions & Write-Up

**Task**
- Write a 150–250 word summary covering: which model you chose and why (including whether the hand-crafted interaction term mattered), its expected error margin, the top 3–5 growth drivers, one scenario-simulation finding, and one concrete next step.


*(Write your conclusions here.)*